# Solar AI - Ground-level PV defect research detector (YOLO11n)

Trains `ground_yolo11n_baseline_v1` on a Kaggle GPU. **3-class first experiment:**
`0 = panel`, `1 = cracked`, `2 = dusty`.

- **Separate research model** - does NOT touch / replace the production aerial detector
  `weights/yolo_solar.pt`. It is not an aerial panel detector.
- Base model: pretrained **YOLO11n** COCO checkpoint (never the aerial Solar AI model).
- Dataset: the audited + remediated **leakage-clean** Roboflow *panel solar* v1
  (`susan-ifblr/panel-solar-bw945`, CC BY 4.0 as displayed). 6,492 images; whole
  near-duplicate families assigned to one split each (train 4,546 / valid 1,297 / test 649).
- `bird_drop` is EXCLUDED (audit: 9 images / ~3 source photos) - a data-availability
  decision, not a product limitation. **Do not report bird_drop performance.**
- Hotspot is out of scope: RGB cannot validate thermal-hotspot semantics.
- imgsz=640, epochs=100, patience=20, seed=42.

Attach the dataset **`<username>/solar-ai-ground-pv-research-v1`** in the sidebar and set
Accelerator = GPU before running.

## 1. Environment

In [ ]:
import sys, platform, subprocess, json, os, time, hashlib, shutil
from pathlib import Path
print('Python :', platform.python_version())

### CUDA - fail closed if unavailable
Kaggle's free GPU pool can hand out a Pascal **Tesla P100 (sm_60)**. We pin
`torch==2.7.1` / `torchvision==0.22.1` (last line retaining Pascal support, CUDA 12.6
wheels) - the pins proven on this account's prior runs.

In [ ]:
PINNED = ['torch==2.7.1', 'torchvision==0.22.1', 'ultralytics==8.4.138', 'PyYAML>=6.0.1']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINNED], check=True)

import torch
assert torch.cuda.is_available(), 'CUDA is not available - this notebook requires a Kaggle GPU (Settings -> Accelerator -> GPU).'
print('torch      :', torch.__version__)
print('CUDA       :', torch.version.cuda)
print('device     :', torch.cuda.get_device_name(0))
print('capability :', torch.cuda.get_device_capability(0))
props = torch.cuda.get_device_properties(0)
print('memory GB  :', round(props.total_memory / 1e9, 1))
import ultralytics; print('ultralytics:', ultralytics.__version__)

## 2. Locate the attached Kaggle dataset + verify provenance

In [ ]:
candidates = list(Path('/kaggle/input').rglob('data.yaml'))
assert candidates, 'No data.yaml under /kaggle/input - attach the dataset in the sidebar first.'
DATA_YAML = None
for c in candidates:
    if 'ground-pv-research' in str(c) or 'ground_pv' in str(c).lower():
        DATA_YAML = c; break
SRC_DATA_YAML = DATA_YAML or candidates[0]
DATA_ROOT = SRC_DATA_YAML.parent
print('mounted data.yaml :', SRC_DATA_YAML)
print(SRC_DATA_YAML.read_text())

# Ultralytics resolves a relative `path:` against its own datasets dir / CWD,
# NOT against the yaml's location - so on the read-only Kaggle mount we must
# write a copy with an ABSOLUTE path. images/ and labels/ live next to it.
import yaml as _yaml
_cfg = _yaml.safe_load(SRC_DATA_YAML.read_text())
_cfg['path'] = str(DATA_ROOT.resolve())
_cfg['train'], _cfg['val'], _cfg['test'] = 'images/train', 'images/valid', 'images/test'
DATA_YAML = Path('/kaggle/working/ground_pv_data.yaml')
DATA_YAML.write_text(_yaml.safe_dump(_cfg, sort_keys=False))
print('training data.yaml (absolute):', DATA_YAML)
print(DATA_YAML.read_text())
for _s in ('images/train', 'images/valid', 'images/test', 'labels/train', 'labels/valid', 'labels/test'):
    assert (DATA_ROOT / _s).is_dir(), f'missing {_s} under {DATA_ROOT}'
print('all image/label split dirs present under the mount')

# provenance shipped inside the dataset
prov = DATA_ROOT / 'provenance'
if prov.is_dir():
    rr = prov / 'remediation_report.json'
    if rr.is_file():
        r = json.loads(rr.read_text())
        print('remediation: no_group_crosses_splits =', r.get('no_group_crosses_splits'),
              ' residual_cross_split_near_dup_pairs =', r.get('residual_cross_split_near_dup_pairs'))
        assert r.get('no_group_crosses_splits') is True
        assert r.get('residual_cross_split_near_dup_pairs') == 0
    print('provenance files:', sorted(p.name for p in prov.iterdir()))
else:
    print('WARNING: no provenance/ directory in the dataset')

## 3. Validate data.yaml + dataset statistics (no training yet)

In [ ]:
import yaml
cfg = yaml.safe_load(DATA_YAML.read_text())
names = cfg['names']
if isinstance(names, list): names = {i: n for i, n in enumerate(names)}
names = {int(k): v for k, v in names.items()}
EXPECTED = {0: 'panel', 1: 'cracked', 2: 'dusty'}
assert names == EXPECTED, f'class taxonomy mismatch: {names} != {EXPECTED}'
assert 'bird_drop' not in names.values(), 'bird_drop must not be present'
print('class taxonomy OK:', names)

from collections import Counter
for split_key in ('train', 'val', 'test'):
    rel = cfg.get(split_key)
    if not rel: continue
    img_dir = (DATA_ROOT / rel)
    lbl_dir = Path(str(img_dir).replace('/images', '/labels'))
    imgs = [p for p in img_dir.glob('*') if p.suffix.lower() in {'.jpg','.jpeg','.png','.webp'}]
    inst, img_with = Counter(), Counter()
    for im in imgs:
        lp = lbl_dir / (im.stem + '.txt')
        if not lp.is_file(): continue
        seen = set()
        for line in lp.read_text().splitlines():
            p = line.split()
            if len(p) == 5:
                cid = int(p[0]); inst[EXPECTED[cid]] += 1; seen.add(EXPECTED[cid])
        for s in seen: img_with[s] += 1
    print(f'  {split_key:5s}  images={len(imgs):5d}  instances={dict(inst)}  images_with={dict(img_with)}')

## 4. Train `ground_yolo11n_baseline_v1`
Base model `yolo11n.pt` (COCO-pretrained). Clean baseline only. Resolved args printed first.

In [ ]:
from ultralytics import YOLO

PROJECT = '/kaggle/working/research_runs/ground_pv'
RUN_NAME = 'ground_yolo11n_baseline_v1'
BASE_MODEL = 'yolo11n.pt'
SEED = 42

train_args = dict(
    data=str(DATA_YAML), epochs=100, imgsz=640, patience=20, seed=SEED,
    batch=-1, device=0, project=PROJECT, name=RUN_NAME, exist_ok=True,
    pretrained=True, cos_lr=True, optimizer='auto', verbose=True, val=True,
    # conservative augmentation (must not distort defect semantics)
    hsv_h=0.015, hsv_s=0.4, hsv_v=0.4, degrees=5.0, translate=0.1, scale=0.3,
    shear=0.0, perspective=0.0, flipud=0.0, fliplr=0.5, mosaic=0.5, mixup=0.0, copy_paste=0.0,
)
print(json.dumps({k: str(v) for k, v in train_args.items()}, indent=2))

t0 = time.time()
model = YOLO(BASE_MODEL)
results = model.train(**train_args)
train_seconds = time.time() - t0
RUN_DIR = Path(results.save_dir)
print('run dir:', RUN_DIR, ' elapsed(s):', round(train_seconds))

## 5. Evaluate - validation and the HELD-OUT leakage-clean test set, per-class

In [ ]:
best = RUN_DIR / 'weights' / 'best.pt'
assert best.is_file(), best
eval_model = YOLO(str(best))

def evaluate(split):
    m = eval_model.val(data=str(DATA_YAML), split=split, imgsz=640, iou=0.5, plots=True, verbose=False)
    box = m.box
    per_class = {}
    for pos, cidx in enumerate(list(box.ap_class_index)):
        p, r, ap50, ap = box.class_result(pos)
        per_class[EXPECTED[int(cidx)]] = dict(precision=float(p), recall=float(r),
                                              AP50=float(ap50), AP50_95=float(ap))
    return dict(
        overall=dict(precision=float(box.mp), recall=float(box.mr),
                     mAP50=float(box.map50), mAP50_95=float(box.map)),
        per_class=per_class, save_dir=str(m.save_dir))

evaluation = {}
for split in ('val', 'test'):
    evaluation[split] = evaluate(split)
    print(split, json.dumps(evaluation[split]['overall'], indent=2))
    print('  per-class:', json.dumps(evaluation[split]['per_class'], indent=2))

## 6. Package artifacts + training manifest + SHA-256

In [ ]:
OUT = Path('/kaggle/working/output'); OUT.mkdir(exist_ok=True)
for name in ('best.pt', 'last.pt'):
    src = RUN_DIR / 'weights' / name
    if src.is_file(): shutil.copy2(src, OUT / name)
for pat in ('results.csv','results.png','args.yaml',
            'confusion_matrix.png','confusion_matrix_normalized.png',
            'BoxPR_curve.png','BoxF1_curve.png','BoxP_curve.png','BoxR_curve.png',
            'PR_curve.png','F1_curve.png','P_curve.png','R_curve.png',
            'labels.jpg','labels_correlogram.jpg'):
    for f in RUN_DIR.rglob(pat): shutil.copy2(f, OUT / f.name)
# copy the val/ and test/ eval dirs (confusion matrix + curves for each split)
for sd in {e['save_dir'] for e in evaluation.values()}:
    sd = Path(sd)
    if sd.is_dir(): shutil.copytree(sd, OUT / ('eval_' + sd.name), dirs_exist_ok=True)
pred_dir = OUT / 'predictions'; pred_dir.mkdir(exist_ok=True)
test_imgs = sorted((DATA_ROOT / cfg['test']).glob('*'))[:16]
if test_imgs:
    YOLO(str(best)).predict(source=[str(p) for p in test_imgs], conf=0.25, save=True,
                            project=str(pred_dir), name='test_examples', exist_ok=True)

sha = hashlib.sha256((OUT / 'best.pt').read_bytes()).hexdigest()
git_sha = None
try: git_sha = subprocess.check_output(['git','-C','/kaggle/working/solar-ai-framework','rev-parse','HEAD'], text=True).strip()
except Exception: pass
prov_summary = {}
try: prov_summary = json.loads((DATA_ROOT / 'provenance' / 'remediation_report.json').read_text())
except Exception: pass

import csv as _csv
results_rows = list(_csv.DictReader(open(RUN_DIR / 'results.csv')))
best_epoch = None
if results_rows:
    key = next((k for k in results_rows[0] if 'mAP50-95' in k and 'B' in k), None)
    if key:
        best_epoch = max(range(len(results_rows)), key=lambda i: float(results_rows[i][key])) + 1

manifest = dict(
    experiment_name=RUN_NAME,
    dataset_name='panel solar (Roboflow) - audited + remediated leakage-clean 3-class',
    dataset_version=1,
    dataset_source='https://universe.roboflow.com/susan-ifblr/panel-solar-bw945',
    dataset_license='CC BY 4.0 (as displayed by Roboflow)',
    classes=EXPECTED, bird_drop='EXCLUDED (data availability - not a product limitation)',
    model=BASE_MODEL, pretrained_source='YOLO11n COCO-pretrained (Ultralytics)',
    epochs=train_args['epochs'], imgsz=train_args['imgsz'], batch=train_args['batch'],
    patience=train_args['patience'], optimizer='auto', cos_lr=True, seed=SEED,
    augmentation={k: train_args[k] for k in ('hsv_h','hsv_s','hsv_v','degrees','translate',
        'scale','shear','perspective','flipud','fliplr','mosaic','mixup','copy_paste')},
    device='cuda:0', gpu=torch.cuda.get_device_name(0),
    gpu_memory_gb=round(torch.cuda.get_device_properties(0).total_memory/1e9, 1),
    python_version=platform.python_version(), torch_version=torch.__version__,
    torchvision_version=__import__('torchvision').__version__,
    ultralytics_version=ultralytics.__version__,
    train_seconds=train_seconds, best_epoch=best_epoch, run_dir=str(RUN_DIR),
    val_metrics=evaluation.get('val', {}).get('overall'),
    test_metrics=evaluation.get('test', {}).get('overall'),
    per_class=dict(val=evaluation.get('val', {}).get('per_class'),
                   test=evaluation.get('test', {}).get('per_class')),
    checkpoint_sha256=sha, git_commit_if_available=git_sha,
    dataset_provenance=dict(
        no_group_crosses_splits=prov_summary.get('no_group_crosses_splits'),
        residual_cross_split_near_dup_pairs=prov_summary.get('residual_cross_split_near_dup_pairs'),
        final_split_counts=prov_summary.get('final_split_counts'),
        family_grouping=prov_summary.get('family_grouping')),
)
(OUT / 'training_manifest.json').write_text(json.dumps(manifest, indent=2, default=str))
(OUT / 'evaluation.json').write_text(json.dumps(evaluation, indent=2, default=str))
print('best.pt sha256:', sha)
print(json.dumps(manifest, indent=2, default=str))
print('\nArtifacts:', sorted(p.name for p in OUT.iterdir()))

## 7. Retrieve
Download `/kaggle/working/output/`. Locally save `best.pt` as
`weights/yolo_solar_ground_research_v1.pt` - **never** overwrite `weights/yolo_solar.pt`.
Then run `scripts/evaluate_ground_pv.py` and `scripts/infer_ground_pv.py`, and record the
SHA-256 in `datasets/ground_pv_research/reports/ground_research_model_manifest.json`.

The ground model is a **research artifact only**. It does not replace aerial detection and
makes no claim about RGB hotspot or bird_drop.